In [1]:
!pip install -q langchain langchain-groq gspread

In [2]:
import os
from google.colab import userdata, auth

os.environ["GROQ_KEY2"] = userdata.get("GROQ_KEY2")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq connected successfully!")

Groq connected successfully!


In [4]:
from google.colab import auth
auth.authenticate_user()

In [5]:
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

print("Google Sheets connected successfully!")

Google Sheets connected successfully!


In [6]:
sh = gc.create("Campus Stationery Orders")
ws = sh.sheet1

ws.append_row([
    "name",
    "item",
    "qty",
    "date",
    "total"
])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1fN1Q_eapj_ijc48_UAhEH5mOaN9vPMH08q904dI7_y8


In [7]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

print("Tool libraries imported successfully!")

Tool libraries imported successfully!


In [8]:
MENU = {
    "notebook": 60,
    "pen": 10,
    "pencil": 5,
    "school bag": 500,
    "eraser": 5
}

HOURS = {}

print("Products loaded successfully!")
print("Available products:", ", ".join(MENU.keys()))

Products loaded successfully!
Available products: notebook, pen, pencil, school bag, eraser


In [9]:
class Order(BaseModel):
    """A stationery order taken from a customer's chat message."""

    name: str = Field(description="Customer's first name")

    item: Literal[
        "notebook",
        "pen",
        "pencil",
        "school bag",
        "eraser"
    ] = Field(description="Stationery product")

    qty: int = Field(
        description="Quantity of the product",
        ge=1
    )

    date: str = Field(
        description="Pickup date as the customer said it"
    )

print("Order model created successfully!")

Order model created successfully!


In [10]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a stationery product.

    Args:
        item: Product name, e.g. notebook, pen, pencil, school bag or eraser.
        qty: Quantity required.
    """

    item = item.lower().strip()

    price = MENU.get(item)

    if price is None:
        return f"'{item}' is not available in our product list. We have: {', '.join(MENU)}."

    if qty < 1:
        return "Quantity must be at least 1."

    return f"{qty} x {item} = Rs {price * qty}"

print("Price checking tool created successfully!")

Price checking tool created successfully!


In [11]:
@tool
def is_open(day: str) -> str:
    """Check the stationery store's opening hours for a day of the week."""

    day = day.lower().strip()

    hours = HOURS.get(day)

    if hours:
        return f"Open {hours} on {day}."

    return f"Opening hours for {day} are not available. I'll check with the store staff."

print("Store timing tool created successfully!")

Store timing tool created successfully!


In [12]:
@tool
def log_order(name: str, item: str, qty: int, date: str) -> str:
    """Save a confirmed stationery order to the Google Sheets Orders sheet.

    Use only when the customer has provided their name, product, quantity and pickup date.
    """

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date
        )

    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        total
    ])

    return (
        f"Saved: {order.qty} x {order.item} "
        f"for {order.name} on {order.date}, Rs {total}."
    )

print("Order saving tool created successfully!")

Order saving tool created successfully!


In [13]:
SYSTEM = """ROLE:
You are a friendly and helpful AI assistant for Campus Stationery Store.

PRODUCTS AND PRICES:
Notebook = Rs 60.
Pen = Rs 10.
Pencil = Rs 5.
School Bag = Rs 500.
Eraser = Rs 5.

FACTS:
Orders need 15 minutes to prepare.
Opening hours are not currently available.

RULES:
1. Always use the given product information.
2. Always use get_price to calculate product prices.
3. Always use is_open when customers ask about store timings.
4. Never invent prices, products or opening hours.
5. If information is unavailable, say "I'll check with the store staff."
6. Before using log_order, collect the customer's name, product, quantity and pickup date.
7. Save an order only when the customer clearly asks to place it.
8. After saving, read the order back with the total price.
9. Never promise an exact delivery time.
10. Do not offer discounts unless included in the facts.

FORMAT:
Under 60 words.
Friendly English or Telugu matching the customer.
"""

print("System prompt created successfully!")

System prompt created successfully!


In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

print("Campus Stationery Store AI Agent created successfully!")

Campus Stationery Store AI Agent created successfully!


In [15]:
history = []

def chat(text: str) -> str:
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print("Chat function created successfully!")

Chat function created successfully!


In [16]:
print(chat("What products are available?"))

print(chat("How much are 5 notebooks?"))

print(chat("Are you open on Sunday?"))

Here’s what we have in stock:  
- Notebook – Rs 60  
- Pen – Rs 10  
- Pencil – Rs 5  
- School Bag – Rs 500  
- Eraser – Rs 5  

Let me know what you’d like!
5 notebooks cost Rs 300. Let me know if you’d like to order!
Opening hours for Sunday are not available. I’ll check with the store staff and get back to you!


In [17]:
print(chat("My name is Ravi. Please place an order for 5 notebooks for 30 November."))

✅ Order saved!  
Ravi, you’ve ordered 5 notebooks for 30 November. Total: Rs 300.  
We’ll prepare it in 15 minutes. Let me know if you need anything else!


In [18]:
print(chat("What is the total price?"))

print(chat("Show me my order details."))

Your total is Rs 300 for the 5 notebooks. Let me know if you’d like to add anything else!
Here’s your order:  
- 5 notebooks  
- Pickup date: 30 November  
- Total: Rs 300  

Let me know if anything needs changing!


In [19]:
print(chat("What is the total price?"))

print(chat("Show me my order details."))

Your total is Rs 300 for the 5 notebooks. Let me know if you’d like to add anything else!
**Your order details**  
- 5 notebooks  
- Pickup date: 30 November  
- Total: Rs 300  

Let me know if you’d like to make any changes!


In [20]:
records = ws.get_all_records()

for record in records:
    print(record)

{'name': 'Ravi', 'item': 'notebook', 'qty': 5, 'date': '30 November', 'total': 300}
